# Nisjemodell — vektor-overlegg (03b)

Leser det generaliserte NiN-laget fra **`03a_nisjemodell_raster.ipynb`** og legger
på vektor-overleggene:

tørrfall → observasjons-substrat → klipp til grunnlinje → trekk fra land →
**checkpoint (parquet)** → Naturkart-attributter → lagre.

Checkpointet etter land-subtraksjon gjør at man kan kjøre attributt-steget på nytt
(reload-cellen) uten å kjøre overleggene om igjen.

In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import mnk
import mnk.attributes
import mnk.utils
import mnk.vectorize
import mnk.rules

CRS = "EPSG:25833"
EPSG = 25833
NODATA = 255

## Kilder og stier

In [ ]:
REPO = Path.cwd()

# Inngang: generalisert NiN-lag fra 03a.
nin_vec_path = REPO / "nisjedata-modell_norge_2026_25833_generalized.gpkg"

# Observasjons-substrat (autoritativt, vektor-overlay).
obs_substrat_parquet = (
    "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-klassifisering_norge_latest_25833.geo.parquet"
)
# Tørrfall med innelukket vann (prepped av mnk.sources.prepare_torrfall_filled).
torrfall_parquet = "gs://niva-geodata/MarintNaturKart/results/torrfall-filled_norge_25833.geo.parquet"
# 1=DEM, 0=no DEM; per-polygon majoritet -> sannsynlighet_beskrivelse.
dem_path = "gs://niva-geodata/MarintNaturKart/features/norge_is_dem.tif"
land_parquet = "gs://niva-geodata/MarintNaturKart/aux/land-mask-norway_FKB-osm-seaborder-subdivided-fullres_v3_25833_20260924.geo.parquet"

# Checkpoint (kun parquet) etter klipp/land, før attributter.
clipped_checkpoint = REPO / "nisjemodell_norge_2026_25833_clipped.geo.parquet"

## NiN-koding

`nin_int` (1-12) kodes først; `nin_kode_fkid`/`NiN_name` utledes **én gang** rett
før checkpointet (via `set_nin_codes`, to skalar-`map` i stedet for `apply`).

In [ ]:
# NiN integer (1-12) -> (nin_kode_fkid, NiN_name), fra mnk.rules.
NIN_INT_MAP = {
    1: mnk.rules.classify_nisje("eufotisk", "brakk", "løst", "innenfor"),   # NA-MC04
    2: mnk.rules.classify_nisje("eufotisk", "salt",  "løst", "innenfor"),   # NA-MA04
    3: mnk.rules.classify_nisje("eufotisk", "brakk", "fast", "innenfor"),   # NA-MC01
    4: mnk.rules.classify_nisje("eufotisk", "salt",  "fast", "innenfor"),   # NA-MA01
    5: mnk.rules.classify_nisje("eufotisk", "brakk", "løst", "utenfor"),    # NA-MC04
    6: mnk.rules.classify_nisje("afotisk",  "brakk", "løst", "utenfor"),    # NA-MC04
    7: mnk.rules.classify_nisje("eufotisk", "salt",  "løst", "utenfor"),    # NA-MA05
    8: mnk.rules.classify_nisje("afotisk",  "salt",  "løst", "utenfor"),    # NA-MA06
    9: mnk.rules.classify_nisje("eufotisk", "brakk", "fast", "utenfor"),    # NA-MC02
    10: mnk.rules.classify_nisje("afotisk", "brakk", "fast", "utenfor"),    # NA-MC02
    11: mnk.rules.classify_nisje("eufotisk", "salt", "fast", "utenfor"),    # NA-MA02
    12: mnk.rules.classify_nisje("afotisk", "salt", "fast", "utenfor"),     # NA-MA03
}
NIN_CODE = {k: v[0] for k, v in NIN_INT_MAP.items()}
NIN_NAME = {k: v[1] for k, v in NIN_INT_MAP.items()}

def set_nin_codes(gdf):
    """Derive nin_kode_fkid/NiN_name from nin_int (two vectorised maps)."""
    gdf["nin_kode_fkid"] = gdf["nin_int"].map(NIN_CODE)
    gdf["NiN_name"] = gdf["nin_int"].map(NIN_NAME)
    return gdf

# Only euphotic "outside" codes change inside tørrfall (intertidal/euphotic).
TORRFALL_REMAP = {5: 1, 7: 2, 9: 3, 11: 4}

# Observation overlay swaps ONLY the substrate axis of an existing NiN code.
SUBSTRAT_TO_FAST = {1: 3, 2: 4, 5: 9, 6: 10, 7: 11, 8: 12}
SUBSTRAT_TO_LOST = {v: k for k, v in SUBSTRAT_TO_FAST.items()}

BUNN_TO_INT = {"løsbunn": 0, "blanding": 0, "fastbunn": 1}  # 0=soft, 1=hard

## Last NiN-lag (fra 03a) + slå sammen slivere

In [ ]:
gdf_nin = gpd.read_file(nin_vec_path)
gdf_nin = gdf_nin.rename(columns={"class_int": "nin_int"})
gdf_nin = gdf_nin[gdf_nin["nin_int"] != NODATA].reset_index(drop=True)
gdf_nin = gdf_nin.explode(index_parts=False).reset_index(drop=True)
if gdf_nin.crs is None or gdf_nin.crs.to_epsg() != EPSG:
    gdf_nin = gdf_nin.set_crs(CRS)

# Merge thin generalization slivers into their largest same-class neighbour
# BEFORE the vector overlays, so slivers are not propagated and overlays run faster.
gdf_nin = mnk.vectorize.merge_slivers(gdf_nin, min_area=2500, max_thinness=0.1, class_col="nin_int")
# Majority DEM coverage per polygon, carried as a temp flag to sannsynlighet_beskrivelse.
gdf_nin["_is_dem"] = mnk.utils.dem_majority(gdf_nin, dem_path)

# nin_kode_fkid/NiN_name utledes først ved checkpointet (ikke nødvendig her).
print(f"Polygoner: {len(gdf_nin):,}")
gdf_nin.head()

## Last observasjons-substrat (nasjonalt)

Substrat for selve gridet hentes fra det modellerte rasteret i loopen under.
Her lastes kun **observasjons**-polygonene (`nisjedata-substrat-klassifisering`),
som legges på som vektor-overlay etter vektoriseringen for å bevare geometrien.


In [ ]:
BUNN_TO_INT = {"løsbunn": 0, "blanding": 0, "fastbunn": 1}  # 0=soft, 1=hard

gdf_obs = gpd.read_parquet(obs_substrat_parquet).to_crs(CRS)
gdf_obs = gdf_obs[gdf_obs["BunnType"].isin(BUNN_TO_INT)].reset_index(drop=True)
gdf_obs["_obs_fast"] = gdf_obs["BunnType"].map(BUNN_TO_INT).astype("uint8")
gdf_obs = gdf_obs[["geometry", "_obs_fast"]]
_ = gdf_obs.sindex  # build spatial index
print(f"Observasjons-polygoner: {len(gdf_obs):,}")

## Tørrfall i vektor-steget

Tørrfallspolygonene er små og detaljerte. I stedet for å rasterisere dem til 50 m
legger vi dem på som vektor-overlay: NiN-polygoner (eufotiske) som overlapper tørrfall
splittes, og de delene som ligger **innenfor** tørrfall remappes:

`NA-MA05→NA-MA04`, `NA-MC02→NA-MC01`, `NA-MA02→NA-MA01`, `NA-MC04→NA-MC04`.

Tørrfallspolygonene følger ikke landmasken eksakt. Vannmasken (som følger landlinjen) klippes
derfor én gang på forhånd med tørrfall i `mnk.sources.prepare_torrfall_filled`: vannbiter som
blir frittstående (avskåret fra åpent hav mellom tørrfall og land) slås inn i tørrfall. Åpent
hav er sammenhengende og blir aldri frittstående, så tørrfall vokser aldri mot sjø.

In [ ]:
# Tørrfall with enclosed water between tørrfall and land merged in, prepared
# once with mnk.sources.prepare_torrfall_filled().
gdf_torrfall = gpd.read_parquet(torrfall_parquet).to_crs(CRS)[["geometry"]].reset_index(drop=True)
print(f"Tørrfall-polygoner: {len(gdf_torrfall):,}")

In [ ]:
# Only codes that tørrfall actually changes need splitting
remap_mask = gdf_nin["nin_int"].isin(TORRFALL_REMAP)
gdf_remap = gdf_nin[remap_mask].reset_index(drop=True)
gdf_keep = gdf_nin[~remap_mask].reset_index(drop=True)

# Inside tørrfall: intersect and remap the NiN code
inside = mnk.vectorize.intersect_layers(gdf_remap, gdf_torrfall)
inside["nin_int"] = inside["nin_int"].map(TORRFALL_REMAP).astype(gdf_remap["nin_int"].dtype)

# Outside tørrfall: difference, code unchanged
outside = mnk.vectorize.subtract_layer(gdf_remap, gdf_torrfall)

gdf_nin_tf = pd.concat([gdf_keep, outside, inside], ignore_index=True)
gdf_nin_tf = gpd.GeoDataFrame(gdf_nin_tf, geometry="geometry", crs=CRS)
gdf_nin_tf = gdf_nin_tf[~gdf_nin_tf.is_empty].explode(index_parts=False).reset_index(drop=True)
print(f"Polygoner etter tørrfall-overlay: {len(gdf_nin_tf):,}")

# Free intermediates to keep peak memory down
del gdf_nin, gdf_torrfall, gdf_remap, gdf_keep, inside, outside, remap_mask
gc.collect()

## Observasjons-substrat i vektor-steget

Observasjonspolygonene er autoritative for substrat. De legges på som vektor-overlay
slik at de observerte grensene bevares nøyaktig (ingen piksel-snapping):

1. `intersection` med NiN-laget → hver bit arver lys+salinitet fra det underliggende
   polygonet (store observasjoner splittes kun langs lys/salinitet-grenser),
2. kun substrat-aksen byttes via `SUBSTRAT_TO_FAST` / `SUBSTRAT_TO_LOST`,
3. `difference` stanser observasjonsfotavtrykket ut av NiN-laget, og de byttede
   observasjonsbitene limes inn igjen.
4. tilgrensende observasjonsbiter med samme NiN-kode slås sammen (dissolve), slik at
   observasjonene ikke blir unødvendig oppsplittet.


In [ ]:
_dtype = gdf_nin_tf["nin_int"].dtype

# Inherit light+salinity from the underlying NiN polygon; only substrate is swapped
obs_pieces = mnk.vectorize.intersect_layers(gdf_obs, gdf_nin_tf)

In [ ]:
obs_pieces["nin_int"] = obs_pieces["nin_int"].astype("int64")
_fast = obs_pieces["_obs_fast"] == 1
obs_pieces.loc[_fast, "nin_int"] = obs_pieces.loc[_fast, "nin_int"].map(lambda v: SUBSTRAT_TO_FAST.get(v, v))
obs_pieces.loc[~_fast, "nin_int"] = obs_pieces.loc[~_fast, "nin_int"].map(lambda v: SUBSTRAT_TO_LOST.get(v, v))
obs_pieces["nin_int"] = obs_pieces["nin_int"].astype(_dtype)

# Dissolve after the overlay so the stamp-out below still uses the small
# original observation polygons (much faster)
n_before = len(obs_pieces)
obs_pieces = (
    obs_pieces[["nin_int", "geometry"]]
    .dissolve(by="nin_int", as_index=False)
    .explode(index_parts=False)
    .reset_index(drop=True)
)
print(f"Observasjonsbiter: {n_before:,} -> {len(obs_pieces):,} etter sammenslåing")

# Stamp the observation footprint out of the NiN layer, then add the swapped pieces
nin_minus = mnk.vectorize.subtract_layer(gdf_nin_tf, gdf_obs[["geometry"]])

nin_minus["_substrat_kilde"] = "substrat_grunn_modellert"
obs_pieces["_substrat_kilde"] = "substrat_kartlagt_ngu"

gdf_nin_obs = pd.concat([nin_minus, obs_pieces], ignore_index=True)
gdf_nin_obs = gpd.GeoDataFrame(gdf_nin_obs, geometry="geometry", crs=CRS)
gdf_nin_obs = gdf_nin_obs[~gdf_nin_obs.is_empty].explode(index_parts=False).reset_index(drop=True)
# Derive nin_kode_fkid/NiN_name once, here (fast scalar maps instead of apply(pd.Series)).
gdf_nin_obs = set_nin_codes(gdf_nin_obs)
gdf_nin_obs = gdf_nin_obs[["nin_int", "nin_kode_fkid", "NiN_name", "_substrat_kilde", "_is_dem", "geometry"]]
print(f"Polygoner etter observasjons-overlay: {len(gdf_nin_obs):,}")

del gdf_nin_tf, gdf_obs, obs_pieces, nin_minus
gc.collect()

## Trekk fra land

## Klipp til grunnlinje (1 nautisk mil)

Fjern polygoner utenfor grunnlinje + 1 nautisk mil, før land trekkes fra.


In [ ]:
print(f"Polygoner før grunnlinje-klipp: {len(gdf_nin_obs):,}")
gdf_nin_obs = mnk.vectorize.clip_to_grunnlinje(gdf_nin_obs)
print(f"Polygoner etter grunnlinje-klipp: {len(gdf_nin_obs):,}")

In [ ]:
land = gpd.read_parquet(land_parquet)
land = land.to_crs(CRS)

gdf_nin_land = mnk.vectorize.subtract_land(gdf_nin_obs, land)
print(f"Polygoner etter land-subtraksjon: {len(gdf_nin_land):,}")

del land, gdf_nin_obs
gc.collect()

## Checkpoint: klippet lag (parquet)

Lagrer det klippede laget (nin_int + kode/navn + substrat-kilde + DEM-flagg) som
parquet. Kjør **reload**-cellen under for å kjøre attributt-steget på nytt uten å
kjøre overleggene om igjen.

In [ ]:
gdf_nin_land.to_parquet(clipped_checkpoint, compression="snappy")
print(f"Checkpoint lagret: {clipped_checkpoint.name} ({len(gdf_nin_land):,} polygoner)")

## Legg til Naturkart-attributter

Avled datamodell-felt (nin_flate_NK) som kan utledes nå. `nkm_lys`, `nkm_lm-sa`,
`nkm_lm-dk` og `nkm_lm-tv` får kodene fra `KODE k_nkm_<..>` (se `mnk.rules.NKM_*`).

`sannsynlighet_beskrivelse`: `høy` for observert substrat (NGU), `medium` der
vi har DEM-data (majoritet av pikslene i `features/norge_is_dem.tif`),
ellers `lav`.

In [ ]:
# Reload checkpoint (kjør denne for å gjenoppta attributt-steget etter restart).
gdf_nin_land = gpd.read_parquet(clipped_checkpoint)
print(f"Lastet checkpoint: {len(gdf_nin_land):,} polygoner")

In [ ]:
gdf_nin_land = mnk.attributes.add_nkm_attributes(gdf_nin_land)
print(gdf_nin_land["sannsynlighet_beskrivelse"].value_counts())
gdf_nin_land.head()

## Lagre resultat

In [ ]:
fname = mnk.utils.to_filename("nisjemodell", "norge", "2026", CRS.split(":")[1])
out_dir = REPO

# Parquet er pipeline-utdata (leses av notebook 04) – billig og minnegjerrig.
gdf_nin_land.to_parquet(out_dir / f"{fname}.geo.parquet", compression="snappy")
print(f"Lagret {len(gdf_nin_land):,} polygoner -> {fname}.geo.parquet")

# Valgfri GPKG for QGIS-inspeksjon; chunk-skriving holder minnebruken nede.
gpkg_path = out_dir / f"{fname}.gpkg"
gpkg_path.unlink(missing_ok=True)
CHUNK = 100_000
for start in range(0, len(gdf_nin_land), CHUNK):
    gdf_nin_land.iloc[start : start + CHUNK].to_file(
        gpkg_path, driver="GPKG", layer="nin", mode="w" if start == 0 else "a"
    )
print(f"Lagret {len(gdf_nin_land):,} polygoner -> {fname}.gpkg")

In [ ]:
# Upload to PostGIS (optional)
mnk.utils.to_postgis(gdf_nin_land, fname)